# Edmonton Office Brochure Extraction Pipeline
## Google Colab Entry Point

In [ ]:
# Cell 1 — Instalar dependências
!pip install pymupdf pytesseract Pillow rapidfuzz pydantic openpyxl requests python-dotenv pandas
!apt-get install -y tesseract-ocr -q

In [ ]:
# Cell 2 — Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 — Clonar repo e configurar variáveis de ambiente
import os, sys, getpass

REPO_DIR = '/content/brochure-pipeline'

# Sempre parte de /content para evitar aninhamento de pastas
os.chdir('/content')

# Remove clone anterior e clona sempre limpo
!rm -rf {REPO_DIR}
!git clone --branch claude/edmonton-brochure-extraction-2r7Bu \
    https://github.com/dionathan-santos/brochure.git {REPO_DIR}

# Entra no diretório correto (absoluto)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Chave Gemini ──────────────────────────────────────────────────────────
# Tenta via Colab Secrets; se falhar, pede a chave manualmente (campo senha)
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GOOGLE_MAPS_API_KEY')
    print("✅ GEMINI_API_KEY lida dos Secrets")
except Exception as e:
    print(f"⚠️  Secrets indisponível ({type(e).__name__}) — cole a chave manualmente:")
    os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')

# ── Chave Anthropic (fallback opcional) ───────────────────────────────────
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    os.environ['ANTHROPIC_API_KEY'] = ''

# ── Paths no Google Drive ─────────────────────────────────────────────────
os.environ['MASTER_INVENTORY_PATH'] = '/content/drive/MyDrive/Brochures/Master_Inventory.xlsx'
os.environ['AVAILABILITIES_PATH']   = '/content/drive/MyDrive/Brochures/Availabilities.xlsx'
os.environ['OUTPUT_DIR']            = '/content/drive/MyDrive/Brochures/Output'

# Verificação final
key = os.environ.get('GEMINI_API_KEY', '')
print(f'Diretório: {os.getcwd()}')
print(f'GEMINI_API_KEY: {key[:8]}...' if key else '❌ Chave vazia — verifique acima')
print('✅ Pronto para rodar!' if key else '❌ Corrija a chave antes de continuar')

In [ ]:
# Cell 3b — Diagnóstico: verificar se a chave Gemini funciona e quais modelos estão disponíveis
import requests, os

key = os.environ.get('GEMINI_API_KEY', '')
if not key:
    print("❌ GEMINI_API_KEY não está setada — rode a célula 3 primeiro")
else:
    r = requests.get(
        f"https://generativelanguage.googleapis.com/v1beta/models?key={key}",
        timeout=15,
    )
    print(f"Status: {r.status_code}")
    if r.ok:
        models = [m['name'] for m in r.json().get('models', [])]
        flash = [m for m in models if 'flash' in m.lower()]
        print("✅ API acessível. Modelos flash disponíveis:")
        for m in flash:
            print(f"   {m}")
    else:
        print(f"❌ Erro: {r.text[:500]}")
        print()
        print("Se aparecer 'SERVICE_DISABLED' ou 'API not enabled':")
        print("  → Acesse: https://console.cloud.google.com/apis/library/generativelanguage.googleapis.com")
        print("  → Habilite a 'Generative Language API' no projeto associado à chave")
        print("Se aparecer 'API_KEY_INVALID':")
        print("  → Gere uma nova chave em: https://aistudio.google.com/apikey")

In [ ]:
# Cell 4 — Rodar o pipeline
# Ajuste brochure_dir para a pasta no Drive onde estão os PDFs
# Ajuste brokerage para o nome da corretora (ex: CBRE, Colliers, Cushman)
from main import run_pipeline

output_path = run_pipeline(
    brochure_dir='/content/drive/MyDrive/Brochures/CBRE/',
    brokerage='CBRE'
)
print(f'Output saved: {output_path}')